# F12 · EDA de dígitos (exploración)

Notebook de exploración para el Ejercicio 2 (clasificación de dígitos con MLP).
Las figuras para el informe se generan de forma reproducible con `python -m analysis.ej2_eda`
y el diccionario de datos final queda en `docs/datos/digits.md`.
**No se entrena ningún modelo en este notebook.**

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

root = Path.cwd()
while not (root / "CLAUDE.md").exists():
    root = root.parent
os.chdir(root)
sys.path.insert(0, str(root))

from data.loaders import load_digits_csv, file_sha256, DIGITS_IMAGE_SHAPE
from analysis.ej2_eda import class_balance_table, compute_average_digits, pixel_stats, check_duplicates

## 1. Carga de los Datasets

Cargamos `digits.csv` (desarrollo/ajuste), `digits_test.csv` (producción) y `more_digits.csv` (referencia Ej3).

In [ ]:
ds_train = load_digits_csv("datasets/digits.csv")
ds_test = load_digits_csv("datasets/digits_test.csv")
ds_more = load_digits_csv("datasets/more_digits.csv")

print(f"digits.csv:      {ds_train.X.shape[0]} muestras, {ds_train.X.shape[1]} features (sha: {ds_train.meta['sha256'][:8]}...)")
print(f"digits_test.csv: {ds_test.X.shape[0]} muestras, {ds_test.X.shape[1]} features (sha: {ds_test.meta['sha256'][:8]}...)")
print(f"more_digits.csv: {ds_more.X.shape[0]} muestras, {ds_more.X.shape[1]} features (sha: {ds_more.meta['sha256'][:8]}...)")

## 2. Balance de Clases

Verificamos la distribución de dígitos en cada partición.

In [ ]:
df_balance = class_balance_table(ds_train.y, ds_test.y, ds_more.y)
display(df_balance)

# Observación clave: clase 8 en train
count_8_train = df_balance.loc[8, 'train_n']
count_5_train = df_balance.loc[5, 'train_n']
print(f"Muestras de clase 8 en digits.csv: {count_8_train}")
print(f"Muestras de clase 5 en digits.csv: {count_5_train}")

## 3. Dígito Promedio por Clase

Calculamos la imagen media $28 \times 28$ para cada clase en entrenamiento y test.

In [ ]:
avg_tr = compute_average_digits(ds_train.X, ds_train.y)
avg_te = compute_average_digits(ds_test.X, ds_test.y)

fig, axes = plt.subplots(2, 10, figsize=(15, 3.5))
for c in range(10):
    if avg_tr[c] is not None:
        axes[0, c].imshow(avg_tr[c], cmap="viridis", vmin=0, vmax=1)
        axes[0, c].set_title(f"Tr: {c}")
    else:
        axes[0, c].text(0.5, 0.5, "Sin datos", ha="center", va="center", color="red", fontsize=9)
        axes[0, c].set_title(f"Tr: {c}", color="red")
    axes[0, c].axis("off")

    axes[1, c].imshow(avg_te[c], cmap="viridis", vmin=0, vmax=1)
    axes[1, c].set_title(f"Te: {c}")
    axes[1, c].axis("off")

plt.suptitle("Dígitos promedio: Fila 1 = Train (digits.csv), Fila 2 = Test (digits_test.csv)")
plt.tight_layout()
plt.show()

## 4. Análisis de Píxeles y Varianza

Estudiamos qué píxeles tienen varianza nula (bordes siempre negros) y la escala de entrada.

In [ ]:
stats = pixel_stats(ds_train.X)
print(f"Rango global: [{stats['min']}, {stats['max']}]")
print(f"Media global: {stats['mean_global']:.4f}, Desvío global: {stats['std_global']:.4f}")
print(f"Píxeles con varianza nula (bordes): {stats['zero_var_pixels']} / 784 ({stats['zero_var_pct']}%)")

var_map = np.array(stats['var_map'])
plt.figure(figsize=(5, 5))
plt.imshow(var_map, cmap="magma")
plt.title("Mapa de varianza por píxel (28x28)")
plt.colorbar(label="Varianza")
plt.axis("off")
plt.show()

## 5. Detección de Duplicados

Verificamos si existen filas idénticas dentro de cada dataset o solapamiento entre train y test.

In [ ]:
dups = check_duplicates({"train": ds_train.X, "test": ds_test.X, "more": ds_more.X})
print("Duplicados intra-dataset:", dups["intra_dataset"])
print("Solapamiento inter-dataset:", dups["inter_dataset"])

## 6. Conclusiones para el Modelo y Protocolo

1. **Ausencia de clase 8 en train:** `digits.csv` no tiene ningún 8. El modelo predecirá erróneamente los 243 ejemplos del dígito 8 en test, acotando la accuracy en test a un máximo teórico de $(2497 - 243) / 2497 = 90.27\%$.
2. **Desbalance de clase 5:** Hay solo 271 ejemplos de 5 en train ($2.18\%$) frente a $\approx 1500$ de las otras clases presentes. La estratificación en el split de validación es mandatoria.
3. **Normalización:** Los píxeles ya se encuentran normalizados en $[0, 1]$. Hay 97 píxeles de varianza 0 en los bordes, por lo que `normalize: "none"` es la opción adecuada por defecto.
4. **Protocolo de validación:** Holdout estratificado 80/20 con 3 semillas como protocolo principal de desarrollo.